In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/daqtest/Processor/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from scipy.optimize import curve_fit
import datetime
import pandas as pd
import re
from matplotlib.pyplot import cm
import matplotlib as mpl
from dataclasses import dataclass
# from sklearn.linear_model import LinearRegression
import statsmodels.api as sm



sys.path.insert(0,"../src/")
import common.utils as util
import common.d2d as d2d
import common.run_info as run_info
import gain_analysis.gain_processor_hdf5 as gain_processor

# from run_selection_single_channel import RunInfo
# import WaveformProcessor
# import FitSPE

# from common.utils import vec_regex_search


## Settings

In [ ]:
params = {
    # figure
    'figure.figsize': (12, 6),
    'figure.facecolor': 'white',  # make figure background white
    # axes
    'axes.labelsize': 20,
    'axes.linewidth': 2,
    
    'axes.titlesize': 20,
    'axes.grid.which': 'both',  # gridlines at major, minor or both ticks
    # errorbar
    'errorbar.capsize': 4,
    # font
    'font.size': 18,
    'font.family': 'Times New Roman',
    # color
    'image.cmap': 'viridis',
    # legend
    'savefig.bbox': 'tight',
    'legend.fontsize': 20,
    'legend.frameon': False,
    'legend.numpoints': 1,  # only one marker in legend
    # line
    'lines.linestyle': 'solid',
    'lines.linewidth': 2,
    'lines.markeredgewidth': 1,
    'lines.markersize': 8,
    # text
    'mathtext.default': 'regular',
    'savefig.bbox': 'tight',
    'savefig.transparent': False,
    # tick
    'xtick.top': True,  # draw ticks on the top side
    'xtick.direction': 'in',
    'xtick.labelsize': 20,
    'xtick.major.size': 8,
    'xtick.major.width': 1,
    'xtick.minor.size': 4,
    'xtick.minor.visible': False,
    'xtick.minor.width': 1,

    'ytick.right': False,  # draw ticks on the right side
    'ytick.direction': 'in',
    'ytick.labelsize': 20,
    'ytick.major.size': 6,
    'ytick.major.width': 1,
    'ytick.minor.size': 3,
    'ytick.minor.visible': False,
    'ytick.minor.width': 1,
    # GRIDS
    'grid.linestyle': '--',  ## dashed
    'mathtext.default': 'regular',
}
plt.rcParams.update(params)


In [ ]:
def create_color_scheme(color_map: str, array: object, color_range=(0,1), darken=1, reverse=False):
    values = sorted(np.unique(array))
    if reverse:
        values = values[::-1]
        
    cmap = plt.get_cmap(color_map)
    color = cmap(np.linspace(color_range[0], color_range[1], len(values)))
    
    # https://stackoverflow.com/questions/37517587/how-can-i-change-the-intensity-of-a-colormap-in-matplotlib
    color[:,0:3] *= darken
    
    clr = {values[i]: color[i] for i in range(len(values))}
    return clr

### Reading Files

In [ ]:
read_hdf5 = gain_processor.GainProcessor()
info = read_hdf5.read_run_list(read_hdf5.run_list_path)

In [ ]:
print(info.__dict__.keys())

In [ ]:
color_temperature = create_color_scheme("coolwarm_r", 
                                        info.temperature_K,
                                        darken = 0.8,
                                        color_range=(0.2,0.9),
                                        reverse=True)
color_temperature

In [ ]:
color_channel = create_color_scheme("viridis", 
                                    info.channel)

color_voltage = create_color_scheme("hot", 
                                    info.voltage_preamp1_V,
                                    color_range=(0,0.8))

## Plots

### Baseline

#### per channel time evolution

In [ ]:
# per channel evolution

voltage_list = [-46, -47, -48, -49, -50, -51, -52]

for voltage in voltage_list:
    
    fig, ax = plt.subplots()

    mask = (info.voltage_preamp1_V==voltage)
    voltage_info = info.apply_mask(mask)

    # for i, temperature in enumerate(np.unique(info.temperature_K)):
    #     mask = info.temperature_K==temperature
    #     _tmp__selection_l1 = info.apply_mask(mask, inplace=False)
    
    for j, channel in enumerate(np.unique(voltage_info.channel)):
        mask = voltage_info.channel==channel
        channel_info = voltage_info.apply_mask(mask, inplace=False)
        
        # if i == 0:
        ax.errorbar(channel_info.date_time, channel_info.baseline_mean_V, 
            yerr=channel_info.baseline_std_V, 
            label = f"{channel}", 
            fmt="o-", 
            ecolor = color_channel[channel], 
            capsize=3, 
            color= color_channel[channel])
        # else:
        #     plt.errorbar(_tmp__selection_l2.date_time, _tmp__selection_l2.gain, 
        #         yerr=_tmp__selection_l2.gain_err, 
        #         fmt="o-", 
        #         ecolor = "black", 
        #         capsize=3, 
        #         color=color)
        
    for i, tempe in enumerate(np.unique(info.temperature_K)):
        mask = info.temperature_K==tempe
        temp_info = info.apply_mask(mask, inplace=False)
        
        temp_date_time = sorted(temp_info.date_time)
        ax.axvspan(temp_date_time[0], temp_date_time[-1],
                   color = color_temperature[tempe],
                   alpha = 0.5)
        
    
    plt.legend(bbox_to_anchor = (1,1), ncol=2)
    plt.xticks(rotation=45)
    plt.xlabel("Date Time")
    plt.ylabel("Baseline Mean [V]")
    plt.title(f"Time evolution of baseline of all channels; bias voltage at {voltage:.1f} V")

    


In [ ]:
# per channel evolution#

# select only threshold calibration sets
mask = np.zeros(len(info))
for i, tag in enumerate(info.run_tag):
    tmp_mask = 'threshold_calibration' in tag 
    mask[i] = tmp_mask
    
# tmp.apply_mask(mask)
threshold_cal_runs = info.apply_mask(mask)

# voltage_list = [-46, -47, -48, -49, -50, -51, -52]
voltage_list = [-49]

for voltage in voltage_list:
    
    fig, ax = plt.subplots(figsize=(15, 6))

    mask = (threshold_cal_runs.voltage_preamp1_V==voltage)
    voltage_info = threshold_cal_runs.apply_mask(mask)
    
    
    
    for j, channel in enumerate(np.unique(voltage_info.channel)):
        mask = voltage_info.channel==channel
        channel_info = voltage_info.apply_mask(mask, inplace=False)
        
        # if i == 0:
        ax.plot(channel_info.date_time, channel_info.baseline_std_V, "o-",
            label = f"{channel}", 
            color= color_channel[channel])
    
    # for j, comment in enumerate(np.unique(voltage_info.comment)):
    #     if (comment==''):
    #         continue
        
    #     mask = voltage_info.comment==comment
    #     tmp = voltage_info.apply_mask(mask, inplace=False)
        
    #     print(tmp.date_time[0])
    #     print(comment)
    #     comment = str(comment)
    #     ax.axvline(tmp.date_time[0],linestyle="dashed")
    #     ax.text(tmp.date_time[0], 0, comment, rotation='vertical',
    #             horizontalalignment='left')
        
    for i, tempe in enumerate(np.unique(info.temperature_K)):
        mask = info.temperature_K==tempe
        temp_info = info.apply_mask(mask, inplace=False)
        
        temp_date_time = sorted(temp_info.date_time)
        ax.axvspan(temp_date_time[0], temp_date_time[-1],
                   color = color_temperature[tempe],
                   alpha = 0.5)
        
    
    plt.legend(bbox_to_anchor = (1,1), ncol=2)
    plt.xticks(rotation=45)
    plt.xlabel("Date Time")
    plt.ylabel("Baseline Mean [V]")
    plt.title(f"Time evolution of baseline of all channels; bias voltage at {voltage:.1f} V")

    


In [ ]:
# per channel evolution#
fig, ax = plt.subplots(figsize = (6, 25))
ax_top = ax.twiny()

# select only threshold calibration sets
mask = np.zeros(len(info))
for i, tag in enumerate(info.run_tag):
    tmp_mask = 'threshold_calibration' in tag 
    mask[i] = tmp_mask
    
# tmp.apply_mask(mask)
threshold_cal_runs = info.apply_mask(mask)

# sort
threshold_cal_runs_df = threshold_cal_runs.get_df().sort_values('run_id')

# voltage_list = [-46, -47, -48, -49, -50, -51, -52]
run_id_list = []
comment_list = []
baseline_std_list = []
baseline_std_std = []
baseline_mean_list = []
datetime_list = []
runtime_s_list = []
temperature_list = []

# for voltage in voltage_list:
    
#     mask = (threshold_cal_runs.voltage_preamp1_V==voltage)
#     voltage_info = threshold_cal_runs.apply_mask(mask)
    
for j, comment in enumerate(np.unique(threshold_cal_runs.comment)):
    if (comment==''):
        continue
    
    mask = threshold_cal_runs.comment==comment
    tmp = threshold_cal_runs.apply_mask(mask, inplace=False)
    
    date = str(tmp.date_time[0]).split('T')[0]
    run_id = tmp.run_id[0]
    
    run_id_list.append(run_id)
    comment_list.append(date +' ' +comment)
    datetime_list.append(tmp.date_time[0])
    baseline_std_list.append(tmp.baseline_std_V.max())
    baseline_std_std.append(tmp.baseline_std_V.std())
    baseline_mean_list.append(tmp.baseline_mean_V.mean())
    temperature_list.append(tmp.temperature_K.mean())
    
    #select date before and after
    # date_before = tmp.date_time[0] - np.timedelta64(1,'D')
    # date_after = tmp.date_time[0] + np.timedelta64(1,'D')
    
    # mask = threshold_cal_runs.date_time.astype('datetime64[D]') == date_before.astype('datetime64[D]')
    # selection = threshold_cal_runs.apply_mask(mask)
    # baseline_std_daybefore_list.append(selection.baseline_std_V.mean())
    
    # mask = threshold_cal_runs.date_time.astype('datetime64[D]') == date_after.astype('datetime64[D]')
    # selection = threshold_cal_runs.apply_mask(mask)
    # baseline_std_dayafter_list.append(selection.baseline_std_V.mean())
    
    #select runtime for the runs before
    tmp_idx = threshold_cal_runs_df.index[threshold_cal_runs_df['run_id']==run_id].tolist()[0]
    runtime_s = threshold_cal_runs_df.iloc[tmp_idx-1].runtime_s
    runtime_s_list.append(runtime_s)
    
    
comment_list = [x for _,x in sorted(zip(run_id_list,comment_list))]
datetime_list = [x for _,x in sorted(zip(run_id_list,datetime_list))]
baseline_std_list = [x for _,x in sorted(zip(run_id_list,baseline_std_list))]
baseline_std_std = [x for _,x in sorted(zip(run_id_list,baseline_std_std))]
baseline_mean_list = [x for _,x in sorted(zip(run_id_list,baseline_mean_list))]
runtime_s_list = [x for _,x in sorted(zip(run_id_list,runtime_s_list))]
temperature_list = [x for _,x in sorted(zip(run_id_list,temperature_list))]


ax_top.errorbar(baseline_std_list,comment_list,
            xerr = 0,
            fmt = "o-")

ax.plot(temperature_list,comment_list,  "ro-")

# plt.setp(ax.get_xticklabels(), rotation=60, ha="right", rotation_mode="anchor")
# ax.tick_params(axis='x',labelrotation=30)
# ax.set_xticks(ax.get_xticks())
# ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', rotation_mode='anchor')
ax.yaxis.set_label_position("right")
ax.yaxis.tick_right()

ax_top.set_xlabel("Baseline std [V]")
ax_top.set_ylabel("Comment")
ax.set_xlabel("Temperature [K]")
plt.gca().invert_yaxis()
ax_top.grid(which="both",axis="both")


In [ ]:
# per channel evolution#
fig, ax = plt.subplots(figsize = (6, 25))
ax_top = ax.twiny()

# select only threshold calibration sets
mask = np.zeros(len(info))
for i, tag in enumerate(info.run_tag):
    tmp_mask = 'threshold_calibration' in tag 
    mask[i] = tmp_mask
    
# tmp.apply_mask(mask)
threshold_cal_runs = info.apply_mask(mask)

# sort
threshold_cal_runs_df = threshold_cal_runs.get_df().sort_values('run_id')

# voltage_list = [-46, -47, -48, -49, -50, -51, -52]
run_id_list = []
comment_list = []
baseline_std_list = []
baseline_std_std = []
baseline_mean_list = []
datetime_list = []
runtime_s_list = []
temperature_list = []

# for voltage in voltage_list:
    
#     mask = (threshold_cal_runs.voltage_preamp1_V==voltage)
#     voltage_info = threshold_cal_runs.apply_mask(mask)
    
for j, date in enumerate(np.unique(threshold_cal_runs.date_time.astype('datetime64[D]'))):
    
    mask = threshold_cal_runs.date_time.astype('datetime64[D]')==date
    tmp = threshold_cal_runs.apply_mask(mask)
    
    run_id = tmp.run_id[0]
    
    run_id_list.append(run_id)
    comment_list.append(str(date) +' ' +tmp.comment[0])
    datetime_list.append(str(date))
    baseline_std_list.append(tmp.baseline_std_V.max())
    baseline_std_std.append(tmp.baseline_std_V.std())
    baseline_mean_list.append(tmp.baseline_mean_V.mean())
    temperature_list.append(tmp.temperature_K.mean())
    
    #select date before and after
    # date_before = tmp.date_time[0] - np.timedelta64(1,'D')
    # date_after = tmp.date_time[0] + np.timedelta64(1,'D')
    
    # mask = threshold_cal_runs.date_time.astype('datetime64[D]') == date_before.astype('datetime64[D]')
    # selection = threshold_cal_runs.apply_mask(mask)
    # baseline_std_daybefore_list.append(selection.baseline_std_V.mean())
    
    # mask = threshold_cal_runs.date_time.astype('datetime64[D]') == date_after.astype('datetime64[D]')
    # selection = threshold_cal_runs.apply_mask(mask)
    # baseline_std_dayafter_list.append(selection.baseline_std_V.mean())
    
    #select runtime for the runs before
    tmp_idx = threshold_cal_runs_df.index[threshold_cal_runs_df['run_id']==run_id].tolist()[0]
    runtime_s = threshold_cal_runs_df.iloc[tmp_idx-1].runtime_s
    runtime_s_list.append(runtime_s)
    
    
comment_list = [x for _,x in sorted(zip(run_id_list,comment_list))]
datetime_list = [x for _,x in sorted(zip(run_id_list,datetime_list))]
baseline_std_list = [x for _,x in sorted(zip(run_id_list,baseline_std_list))]
baseline_std_std = [x for _,x in sorted(zip(run_id_list,baseline_std_std))]
baseline_mean_list = [x for _,x in sorted(zip(run_id_list,baseline_mean_list))]
runtime_s_list = [x for _,x in sorted(zip(run_id_list,runtime_s_list))]
temperature_list = [x for _,x in sorted(zip(run_id_list,temperature_list))]


ax_top.errorbar(baseline_std_list,comment_list,
            xerr = 0,
            fmt = "o-")

ax.plot(temperature_list,comment_list,  "ro-")

# plt.setp(ax.get_xticklabels(), rotation=60, ha="right", rotation_mode="anchor")
# ax.tick_params(axis='x',labelrotation=30)
# ax.set_xticks(ax.get_xticks())
# ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', rotation_mode='anchor')
ax.yaxis.set_label_position("right")
ax.yaxis.tick_right()

ax_top.set_xlabel("Baseline std [V]")
ax_top.set_ylabel("Comment")
ax.set_xlabel("Temperature [K]")
plt.gca().invert_yaxis()
ax_top.grid(which="both",axis="both")


#### channel 2

In [ ]:
# per channel evolution#
fig, ax = plt.subplots(figsize = (6, 40))
ax_top = ax.twiny()

# select only threshold calibration sets
mask = np.zeros(len(info))
for i, tag in enumerate(info.run_tag):
    tmp_mask = 'threshold_calibration' in tag 
    mask[i] = tmp_mask
    
# tmp.apply_mask(mask)
threshold_cal_runs = info.apply_mask(mask)

# select only channel 2
mask = threshold_cal_runs.channel==2
tmp = threshold_cal_runs.apply_mask(mask)

# sort
threshold_cal_runs_df = tmp.get_df().sort_values('run_id')

# voltage_list = [-46, -47, -48, -49, -50, -51, -52]
run_id_list = []
comment_list = []
baseline_std_list = []
baseline_std_std = []
baseline_mean_list = []
datetime_list = []
runtime_s_list = []
temperature_list = []

# for voltage in voltage_list:
    
#     mask = (threshold_cal_runs.voltage_preamp1_V==voltage)
#     voltage_info = threshold_cal_runs.apply_mask(mask)
    
# for j, date in enumerate(np.unique(threshold_cal_runs.date_time.astype('datetime64[D]'))):
    
    # mask = threshold_cal_runs_2.date_time.astype('datetime64[D]')==date
    # tmp = threshold_cal_runs_2.apply_mask(mask)

# run_id = tmp.run_id[0]

# date = tmp.date_time[0]
for i, run_id in enumerate(tmp.run_id):
    date = tmp.date_time[i].astype('datetime64[D]')
    run_id_list.append(run_id)
    comment_list.append(str(date) +' ' +tmp.comment[i])
    baseline_std_list.append(tmp.baseline_std_V[i])
    datetime_list.append(str(date))
    # baseline_std_list.append(tmp.baseline_std_V.max())
    # baseline_std_std.append(tmp.baseline_std_V.std())
    # baseline_mean_list.append(tmp.baseline_mean_V.mean())
    temperature_list.append(tmp.temperature_K[i])

#select date before and after
# date_before = tmp.date_time[0] - np.timedelta64(1,'D')
# date_after = tmp.date_time[0] + np.timedelta64(1,'D')

# mask = threshold_cal_runs.date_time.astype('datetime64[D]') == date_before.astype('datetime64[D]')
# selection = threshold_cal_runs.apply_mask(mask)
# baseline_std_daybefore_list.append(selection.baseline_std_V.mean())

# mask = threshold_cal_runs.date_time.astype('datetime64[D]') == date_after.astype('datetime64[D]')
# selection = threshold_cal_runs.apply_mask(mask)
# baseline_std_dayafter_list.append(selection.baseline_std_V.mean())

#select runtime for the runs before
# tmp_idx = threshold_cal_runs_df.index[threshold_cal_runs_df['run_id']==run_id].tolist()[0]
# runtime_s = threshold_cal_runs_df.iloc[tmp_idx-1].runtime_s
# runtime_s_list.append(runtime_s)
    
    
# comment_list = [x for _,x in sorted(zip(run_id_list,comment_list))]
# datetime_list = [x for _,x in sorted(zip(run_id_list,datetime_list))]
# baseline_std_list = [x for _,x in sorted(zip(run_id_list,baseline_std_list))]
# # baseline_std_std = [x for _,x in sorted(zip(run_id_list,baseline_std_std))]
# # baseline_mean_list = [x for _,x in sorted(zip(run_id_list,baseline_mean_list))]
# # runtime_s_list = [x for _,x in sorted(zip(run_id_list,runtime_s_list))]
# temperature_list = [x for _,x in sorted(zip(run_id_list,temperature_list))]


ax_top.errorbar(baseline_std_list,comment_list,
            xerr = 0,
            fmt = "o-")

ax.plot(temperature_list,comment_list,  "ro-")

# plt.setp(ax.get_xticklabels(), rotation=60, ha="right", rotation_mode="anchor")
# ax.tick_params(axis='x',labelrotation=30)
# ax.set_xticks(ax.get_xticks())
# ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', rotation_mode='anchor')
ax.yaxis.set_label_position("right")
ax.yaxis.tick_right()

ax_top.set_xlabel("Baseline std [V]")
ax_top.set_ylabel("Comment")
ax.set_xlabel("Temperature [K]")
plt.gca().invert_yaxis()
ax_top.grid(which="both",axis="both")


In [ ]:
threshold_cal_runs_df.tail(100)

In [ ]:
# per channel evolution#
fig, ax = plt.subplots(figsize = (6, 30))
ax_top = ax.twiny()

# select only threshold calibration sets
mask = np.zeros(len(info))
for i, tag in enumerate(info.run_tag):
    tmp_mask = 'threshold_calibration' in tag 
    mask[i] = tmp_mask
    
# tmp.apply_mask(mask)
threshold_cal_runs = info.apply_mask(mask)

# sort
threshold_cal_runs_df = threshold_cal_runs.get_df().sort_values('run_id')

# voltage_list = [-46, -47, -48, -49, -50, -51, -52]
run_id_list = []
comment_list = []
baseline_std_list = []
baseline_std_std = []
baseline_mean_list = []
datetime_list = []
runtime_s_list = []
temperature_list = []

# for voltage in voltage_list:
    
#     mask = (threshold_cal_runs.voltage_preamp1_V==voltage)
#     voltage_info = threshold_cal_runs.apply_mask(mask)
    
for j, date_time in enumerate(np.unique(threshold_cal_runs.date_time)):
    
    mask = threshold_cal_runs.date_time==date_time
    tmp = threshold_cal_runs.apply_mask(mask)
    
    date = str(tmp.date_time[0]).split('T')[0]
    run_id = tmp.run_id[0]
    
    run_id_list.append(run_id)
    comment_list.append(str(date) +' ' +tmp.comment[0])
    datetime_list.append(str(date))
    baseline_std_list.append(tmp.baseline_std_V.max())
    baseline_std_std.append(tmp.baseline_std_V.std())
    baseline_mean_list.append(tmp.baseline_mean_V.mean())
    temperature_list.append(tmp.temperature_K.mean())
    
    #select date before and after
    # date_before = tmp.date_time[0] - np.timedelta64(1,'D')
    # date_after = tmp.date_time[0] + np.timedelta64(1,'D')
    
    # mask = threshold_cal_runs.date_time.astype('datetime64[D]') == date_before.astype('datetime64[D]')
    # selection = threshold_cal_runs.apply_mask(mask)
    # baseline_std_daybefore_list.append(selection.baseline_std_V.mean())
    
    # mask = threshold_cal_runs.date_time.astype('datetime64[D]') == date_after.astype('datetime64[D]')
    # selection = threshold_cal_runs.apply_mask(mask)
    # baseline_std_dayafter_list.append(selection.baseline_std_V.mean())
    
    #select runtime for the runs before
    tmp_idx = threshold_cal_runs_df.index[threshold_cal_runs_df['run_id']==run_id].tolist()[0]
    runtime_s = threshold_cal_runs_df.iloc[tmp_idx-1].runtime_s
    runtime_s_list.append(runtime_s)
    
    
comment_list = [x for _,x in sorted(zip(run_id_list,comment_list))]
datetime_list = [x for _,x in sorted(zip(run_id_list,datetime_list))]
baseline_std_list = [x for _,x in sorted(zip(run_id_list,baseline_std_list))]
baseline_std_std = [x for _,x in sorted(zip(run_id_list,baseline_std_std))]
baseline_mean_list = [x for _,x in sorted(zip(run_id_list,baseline_mean_list))]
runtime_s_list = [x for _,x in sorted(zip(run_id_list,runtime_s_list))]
temperature_list = [x for _,x in sorted(zip(run_id_list,temperature_list))]


ax_top.errorbar(baseline_std_list,comment_list,
            xerr = 0,
            fmt = "o-")

ax.plot(temperature_list,comment_list,  "ro-")

# plt.setp(ax.get_xticklabels(), rotation=60, ha="right", rotation_mode="anchor")
# ax.tick_params(axis='x',labelrotation=30)
# ax.set_xticks(ax.get_xticks())
# ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', rotation_mode='anchor')
ax.yaxis.set_label_position("right")
ax.yaxis.tick_right()

ax_top.set_xlabel("Baseline std [V]")
ax_top.set_ylabel("Comment")
ax.set_xlabel("Temperature [K]")
plt.gca().invert_yaxis()
ax_top.grid(which="both",axis="both")


In [ ]:
threshold_cal_runs_df = threshold_cal_runs.get_df()
threshold_cal_runs_df.index[threshold_cal_runs_df['run_id']==run_id].tolist()

In [ ]:
threshold_cal_runs_df.iloc[938].run_id

In [ ]:
# select only threshold calibration sets
mask = np.zeros(len(info))
for i, tag in enumerate(info.run_tag):
    tmp_mask = 'threshold_calibration' in tag 
    mask[i] = tmp_mask
threshold_cal_runs = info.apply_mask(mask)
    
# plt.hist2d(df.voltage_preamp1, df.baseline_std_V, bins=[50,50],range=[[0,100],[0,0.001]], cmap='viridis',norm="log")
plt.hist2d(threshold_cal_runs.runtime_s[1:],
           threshold_cal_runs.baseline_std_V[1:],
           cmap='viridis', 
           bins=[100,50],
           range=[[0,50],[0,0.05]],
           norm=mpl.colors.LogNorm())



# plt.plot(info.date_time, info.baseline_std_V,"o")
plt.xlabel("Run time of the previous run [s]")
plt.ylabel("Baseline Std [V]")
plt.xticks(rotation=45)
plt.legend()



In [ ]:
threshold_cal_runs.run_id

#### Mean baseline over all runs and all channels at different temperature vs bias voltage 

In [ ]:

for temperature in np.unique(info.temperature_K):
    gain_list = []
    gain_error_list = []
    voltage_list = []
    # print(temperature)
    mask = info.temperature_K == temperature
    masked_temp = info.apply_mask(mask)
    for voltage in np.unique(masked_temp.voltage_preamp1_V):
        # print(voltage)
        mask = masked_temp.voltage_preamp1_V == voltage
        masked_voltage = masked_temp.apply_mask(mask)
        if len(masked_voltage) > 0:
            gain = masked_voltage.baseline_mean.mean()
            error = masked_voltage.baseline_mean.std()
            gain_list.append(gain)
            gain_error_list.append(error)
            voltage_list.append(voltage)
            
    plt.errorbar(voltage_list, gain_list, 
                 yerr=gain_error_list, 
                 label = f"{temperature} K", 
                 fmt="o", 
                 ecolor = color_temperature[temperature], 
                 capsize=3, 
                 color=color_temperature[temperature])
    # plt.scatter(voltage_list, gain_list, label=f"{temperature} K", 
    #              color=color_temperature[temperature],
    #              edgecolor='k')
plt.legend()
plt.gca().invert_xaxis()
plt.ylabel("Gain")
plt.xlabel("Bias Voltage")
plt.title("Mean over all runs and all channels")


#### Mean baseline over all runs and all channels at different voltage vs temperature 

In [ ]:

for voltage in np.unique(info.voltage_preamp1_V):
    baseline_list = []
    baseline_error_list = []
    temp_list = []
    # print(temperature)
    mask = info.voltage_preamp1_V == voltage
    masked_volt = info.apply_mask(mask)
    for temperature in np.unique(masked_volt.temperature_K):
        # print(voltage)
        mask = masked_volt.temperature_K == temperature
        masked_temp = masked_volt.apply_mask(mask)
        if len(masked_temp) > 0:
            baseline = masked_temp.baseline_mean.mean()
            error = masked_temp.baseline_mean.std()
            baseline_list.append(baseline)
            baseline_error_list.append(error)
            temp_list.append(temperature)
            
    plt.errorbar(temp_list, baseline_list, 
                 yerr=baseline_error_list, 
                 label = f"{voltage} V", 
                 fmt="o-", 
                 ecolor = color_voltage[voltage], 
                 capsize=3, 
                 color=color_voltage[voltage])
    # plt.scatter(voltage_list, gain_list, label=f"{temperature} K", 
    #              color=color_temperature[temperature],
    #              edgecolor='k')
plt.legend()
plt.gca().invert_xaxis()
plt.ylabel("Baseline [V]")
plt.xlabel("Bias Voltage [V]")
plt.title("Mean over all runs and all channels")


## Single-out problematic dataset for further investigation

In [ ]:

# for temperature in np.unique(gain_info.temperature_K):
temperature = 167
voltage = -51
baseline_std_V = 0.012

selected_data = threshold_cal_runs
mask = (selected_data.voltage_preamp1_V == voltage)
mask = mask & (selected_data.temperature_K == temperature)
# mask = mask & (info.channel == channel)
mask = mask & (selected_data.baseline_std_V > baseline_std_V)

tmp = selected_data.apply_mask(mask)

tmp.run_tag

# mask = tmp.run_tag == 'threshold_calibration'

mask = np.zeros(len(tmp))
for i, tag in enumerate(tmp.run_tag):
    tmp_mask = 'threshold_calibration' in tag 
    mask[i] = tmp_mask

print(mask)
# tmp.apply_mask(mask)
tmp.apply_mask(mask, inplace = True)
tmp.bin_full_path


## Threshold problem

In [ ]:
# df = pd.read_csv("../run_info_single_channel.csv", parse_dates=["date_time"],delimiter=",",quotechar='\0')
# df = pd.read_csv("../run_info_single_channel.csv", parse_dates=["date_time"],delimiter=",",quotechar='"', skipinitialspace=True, encoding="utf-8")
# df = pd.read_csv("/home/daqtest/DAQ/SandyAQ/python_wrappers/run_info_single_channel_20240701_2.csv", parse_dates=["date_time"],delimiter=",",quotechar='"', skipinitialspace=True, encoding="utf-8")
df = pd.read_csv("/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/processed/gain_info_single_channel.csv", 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")
df.columns


In [ ]:
info = d2d.data(df)
print(info.__dict__.keys())

mask_record_length_nan = ~np.isnan(info.record_length_sample)
mask_run_tag = utils.vec_regex_search('GXe/gain_calibration', info.run_tag)
mask_run_tag_remove_trash = ~utils.vec_regex_search('trash', info.run_tag)
mask_time = (info.date_time > np.datetime64('2024-05-18'))
mask_start_index_nan = ~np.isnan(info.start_index)
mask_nevents_nan = ~np.isnan(info.number_of_events)

# FIXME: mask channel

mask = mask_run_tag & mask_run_tag_remove_trash & mask_time & mask_record_length_nan & mask_start_index_nan & mask_nevents_nan

# check how much data is removed
info.apply_mask(mask, inplace=True)


In [ ]:
min_bin = min(info.baseline_std_V)
min_bin

In [ ]:
# plt.hist2d(df.voltage_preamp1, df.baseline_std_V, bins=[50,50],range=[[0,100],[0,0.001]], cmap='viridis',norm="log")
min_bin = min(info.baseline_std_V)
max_bin = max(info.baseline_std_V)

# color = cm.viridis(np.linspace(0, 1, len(np.unique(info.voltage_preamp1_V))))

for i, voltage in enumerate(np.unique(info.voltage_preamp1_V)):
    mask = info.voltage_preamp1_V==voltage
    _tmp__selection = info.apply_mask(mask, inplace=False)
    # print(voltage, len(_tmp__selection))
    plt.hist(_tmp__selection.baseline_std_V,
             label=f"{voltage} V", 
             lw=3,
             range=(min_bin,max_bin),
             log=True,
             histtype='step',
             stacked=True, fill=False,
             color=color_voltage[voltage],
             density=True
             )
    
plt.xlabel("Voltage Preamp 1 [V]")
plt.ylabel("Baseline Std [V]")
plt.legend()



In [ ]:
fig, ax_adc = plt.subplots()

threshold_color = "tab:blue"
ax_adc.spines['left'].set_color(threshold_color)
ax_adc.tick_params(axis='y', colors=threshold_color)
ax_adc.yaxis.label.set_color(threshold_color)

color = cm.viridis(np.linspace(0, 1, len(np.unique(info.voltage_preamp1_V))))

ax_baseline = ax_adc.twinx()

# mask = (info.date_time > np.datetime64('2024-05-26'))
# mask = mask & (info.date_time < np.datetime64('2024-06-08'))
# info_time = info.apply_mask(mask, inplace=False)
info_time = info

for i, voltage in enumerate(np.unique(info.voltage_preamp1_V)):
    mask = info_time.voltage_preamp1_V==voltage
    _tmp__selection = info_time.apply_mask(mask, inplace=False)
    ax_baseline.plot(_tmp__selection.date_time, 
                _tmp__selection.baseline_std_V,
                'o',
                color=color_voltage[voltage],
                label = f"{voltage} V")
    
ax_adc.plot(info_time.date_time, 
            info_time.threshold_adc,'.',
            label = "threshold",
            alpha = 0.5)
    
# plt.xlabel("Date time")
# plt.ylabel("Baseline Std [V]")
ax_adc.tick_params(axis='x',labelrotation=30)
ax_adc.set_xlabel('Date Time')
ax_baseline.set_ylabel('Baseline Std [V]')
ax_adc.set_ylabel('Threshold [adc]')

plt.legend(bbox_to_anchor = (1.33,1))


In [ ]:
color = cm.viridis(np.linspace(0, 1, len(np.unique(info.temperature_K))))

# calculate the colors of each entry
# min_temp = min(info.temperature_K)
# max_temp = max(info.temperature_K)
# temp_color = (info.temperature_K - min_temp)/(max_temp-min_temp)*(len(np.unique(info.temperature_K))-1)
# temp_color.astype(int)

for i, temperature in enumerate(np.unique(info.temperature_K)):
    mask = info.temperature_K==temperature
    _tmp__selection = info.apply_mask(mask, inplace=False)
    plt.scatter(_tmp__selection.date_time, 
                _tmp__selection.baseline_std_V,
                color=color_temperature[temperature],
                label = f"{temperature} K")
    
plt.xlabel("Date time")
plt.ylabel("Baseline Mean [V]")
plt.legend(bbox_to_anchor = (1,1))
plt.xticks(rotation=45)

# plt.hist2d(df.voltage_preamp1, df.baseline_std_V, bins=[50,50],range=[[0,100],[0,0.001]], cmap='viridis',norm="log")
# plt.xlabel("Date Time")
# plt.ylabel("Baseline Mean [V]")
# plt.legend()

In [ ]:

for i, temperature in enumerate(np.unique(info.temperature_K)):
    mask = info.temperature_K==temperature
    _tmp__selection_l1 = info.apply_mask(mask, inplace=False)
    for j, channel in enumerate(np.unique(info.channel)):
        mask = _tmp__selection_l1.channel==channel
        _tmp__selection_l2 = _tmp__selection_l1.apply_mask(mask, inplace=False)
        norm = _tmp__selection_l2.baseline_mean_V.mean()
        color = color_temperature[temperature][:3]*((j+1)*0.05)
        
        if i == 0:
            plt.plot(_tmp__selection_l2.date_time, 
                _tmp__selection_l2.baseline_mean_V,".-",
                color=color, label = f"{channel}")
        else:
            plt.plot(_tmp__selection_l2.date_time, 
                _tmp__selection_l2.baseline_mean_V,".-",
                color=color)
    
plt.xlabel("Date time")
plt.ylabel("Baseline Mean [V]")
plt.legend(bbox_to_anchor = (1,1), ncol=2)
plt.xticks(rotation=45)


In [ ]:
mask = ~np.isnan(info.runtime_s)
mask = mask & (info.date_time > np.datetime64('2024-05-26'))
mask = mask & (info.date_time < np.datetime64('2024-06-08'))

_tmp__selection = info.apply_mask(mask,inplace=False)

# plt.hist2d(df.voltage_preamp1, df.baseline_std_V, bins=[50,50],range=[[0,100],[0,0.001]], cmap='viridis',norm="log")
plt.hist2d(_tmp__selection.runtime_s[:-1],
           _tmp__selection.baseline_std_V[1:],
           cmap='viridis', 
           bins=[100,50],
           range=[[0,2000],[0,0.05]],
           norm=mpl.colors.LogNorm())
# plt.plot(info.date_time, info.baseline_std_V,"o")
plt.xlabel("Run time of the previous run [s]")
plt.ylabel("Baseline Std [V]")
plt.xticks(rotation=45)
plt.legend()